# Taller F02 — Benchmarks y evaluación de forecasting

## Contexto

Una empresa registra la **demanda semanal de un componente industrial** durante 120 semanas. El objetivo es evaluar métodos simples de forecasting usando una partición temporal `train/test`.

## Objetivo

Verificar que el estudiante puede:

- construir una partición temporal de entrenamiento y prueba;
- implementar benchmarks mean, naive y drift;
- calcular MAE, RMSE y MASE;
- comparar métodos bajo una misma partición;
- interpretar resultados fuera de muestra.

> No modifique las celdas bloqueadas.


In [25]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

n = 120
semana = np.arange(1, n + 1)

eps = rng.normal(0, 4, n)
demanda = np.zeros(n)
demanda[0] = 100

for t in range(1, n):
    demanda[t] = 100 + 0.55 * (demanda[t-1] - 100) + 0.10 * t + eps[t]

df = pd.DataFrame({"semana": semana, "demanda": demanda})
df.head()


,semana,demanda
0,1,100.000000
1,2,101.062285
2,3,93.198951
3,4,102.142510
4,5,104.131560


## 1. Partición temporal train/test — 2 puntos

Use `h = 12`.

Complete:

- `train`: todas las observaciones anteriores a las últimas 12 semanas;
- `test`: las últimas 12 observaciones;
- `n_train`: número de observaciones de `train`;
- `n_test`: número de observaciones de `test`.


In [26]:
h = 12

train = df.iloc[:-h].copy()
test = df.iloc[-h:].copy()

y_train = train['demanda'].values
y_test = test['demanda'].values

n_train = len(train)
n_test = len(test)

print(f'Observaciones en train: {n_train}')
print(f'Observaciones en test: {n_test}')

Observaciones en train: 108
Observaciones en test: 12


## 2. Benchmarks mean y naive — 2 puntos

Calcule:

- `mean_value`: media de `demanda` en `train`;
- `last_value`: última observación de `demanda` en `train`;
- `forecast_mean`: arreglo de longitud `h`;
- `forecast_naive`: arreglo de longitud `h`.


In [27]:
mean_value = y_train.mean()
last_value = y_train[-1]
forecast_mean = np.full(h, mean_value)
forecast_naive = np.full(h, last_value)

print(f'Media de demanda en train: {mean_value}')
print(f'Ultima observacion de demanda en train: {last_value}')
print(f'Forecast con Mean: \n {forecast_mean}')
print(f'Forecast con Naive \n {forecast_naive}')

Media de demanda en train: 112.07997741610968
Ultima observacion de demanda en train: 126.55123905619861
Forecast con Mean: 
 [112.07997742 112.07997742 112.07997742 112.07997742 112.07997742
 112.07997742 112.07997742 112.07997742 112.07997742 112.07997742
 112.07997742 112.07997742]
Forecast con Naive 
 [126.55123906 126.55123906 126.55123906 126.55123906 126.55123906
 126.55123906 126.55123906 126.55123906 126.55123906 126.55123906
 126.55123906 126.55123906]


## 3. Benchmark drift — 2 puntos

Complete:

- `y_1`: primera observación de `demanda` en `train`;
- `y_T`: última observación de `demanda` en `train`;
- `T`: número de observaciones de `train`;
- `drift_per_week`: cambio promedio por semana;
- `forecast_drift`: pronóstico para los `h` períodos futuros.

Use:

$$
\hat{y}_{T+h|T}
=
y_T+h\left(\frac{y_T-y_1}{T-1}\right)
$$


In [28]:
y_1 = y_train[0]
y_T = y_train[-1]
T = len(y_train)
drift_per_week = (y_T - y_1)/ (T-1)

forecast_drift = y_T + h*drift_per_week

print(f'Primera observacion de demanda en train: {y_1}')
print(f'Última observación de demanda en train: {y_T}')
print(f'Número de observaciones: {T}')
print(f'Cambio promedio por semana de demanda: {drift_per_week}')
print(f'Forecast con Drift: \n {forecast_drift}')

Primera observacion de demanda en train: 100.0
Última observación de demanda en train: 126.55123905619861
Número de observaciones: 108
Cambio promedio por semana de demanda: 0.24814242108596832
Forecast con Drift: 
 129.52894810923024


## 4. MAE y RMSE — 2 puntos

Implemente:

```python
mae(y_true, y_pred)
rmse(y_true, y_pred)
```

Luego calcule MAE y RMSE para mean, naive y drift.


In [41]:
def mae(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.mean(np.abs(y_true - y_pred))

mae_mean = mae(y_test, forecast_mean)
mae_naive = mae(y_test, forecast_naive)
mae_drift = mae(y_test, forecast_drift)

def rmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.sqrt(np.mean(np.abs(y_true - y_pred)**2))

rmse_mean = rmse(y_test, forecast_mean)
rmse_naive = rmse(y_test, forecast_naive)
rmse_drift = rmse(y_test, forecast_drift)

print(f'Mean: \n MAE: {mae_mean} \t RMSE: {rmse_mean}')
print(f'Naive: \n MAE: {mae_naive} \t RMSE: {rmse_naive}')
print(f'Drift: \n MAE: {mae_drift} \t RMSE: {rmse_drift}')

Mean: 
 MAE: 13.272411885803606 	 RMSE: 14.15131373266721
Naive: 
 MAE: 4.295608735532924 	 RMSE: 5.053711886060053
Drift: 
 MAE: 4.928272733186241 	 RMSE: 6.445650204977887


## 5. MASE — 1 punto

Implemente `mase(y_train, y_true, y_pred)` para una serie no estacional.

Si la escala es prácticamente cero, devuelva `np.nan`.

Luego calcule `mase_mean`, `mase_naive` y `mase_drift`.


In [44]:
import numpy as np

def mase(y_train, y_true, y_pred):
    y_train = np.array(y_train)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae_pred = mae(y_true, y_pred)

    denominator = np.mean(np.abs(np.diff(y_train)))

    if denominator == 0 or np.isclose(denominator, 0):
        return np.nan

    return mae_pred / denominator

mase_mean = mase(y_train, y_test, forecast_mean)
mase_naive = mase(y_train, y_test, forecast_naive)
mase_drift = mase(y_train, y_test, forecast_drift)

print(f'Mean: \n MASE: {mase_mean}')
print(f'Naive: \n MASE: {mase_naive}')
print(f'Drift: \n MASE: {mase_drift}')

Mean: 
 MASE: 3.7349209826194776
Naive: 
 MASE: 1.2088051017031887
Drift: 
 MASE: 1.3868398146185366


## 6. Interpretación — 1 punto

En **máximo 150 palabras**, responda:

1. ¿Qué benchmark obtiene el menor MAE?
2. ¿El ranking por RMSE coincide con el ranking por MAE?
3. ¿Cómo interpreta los valores de MASE obtenidos? ¿Alguno es menor que 1?
4. ¿Por qué no sería correcto declarar que un método es “el mejor” sin especificar horizonte, partición y métrica?
5. ¿Cuál benchmark conservaría como referencia para evaluar un modelo más complejo y por qué?


El benchmark que obtiene menor MAE es el Naive, con un valor de 4.2956, además se puede ver que ambos ranking (el RMSE y MAE) coinciden en el orden de los benchmarks. También se ve que ningun valor de MASE es menor a 1, aunque el Naive está muy cerca (1.21), eso significa que este y Drift (1.39) se comportan de manera similar a la línea base de comparación.
Pero, no podemos declarar un método como el mejor sin saber horizonte, partición y métrica debido a que cada uno puede tener su fuerte en diferentes temas, como predecir inmediatamente o a largo plazo, además, los resultados cambian según la cantidad de datos que se usen, y por último, cada métrica penaliza los errores de manera diferente, así que depende de lo que se esté buscando.
Yo usaría como referencia el Naive, ya que fue el mejor en los 3 benchmarks.
